In [0]:
USE CATALOG workspace;
USE SCHEMA ecom_poc;

In [0]:
-- One row per eligible successful payment, already converted to USD.
-- Every revenue number in Gold reads from this view, so the definition lives in one place.
CREATE OR REPLACE VIEW gold_revenue_base
COMMENT 'Eligible revenue payments: SUCCESS txns on non-cancelled, non-returned orders, converted to USD at fixed rates'
AS
SELECT t.txn_id, t.order_id, o.customer_id, o.order_date, o.category, o.country, o.order_status,
       t.amount, t.currency,
       -- Convert to USD; fixed assumed rates: 1 USD = 0.92 EUR = 0.79 GBP = 110 BDT.
       ROUND(CASE t.currency
               WHEN 'USD' THEN t.amount
               WHEN 'EUR' THEN t.amount / 0.92
               WHEN 'GBP' THEN t.amount / 0.79
               WHEN 'BDT' THEN t.amount / 110
             END, 2) AS revenue_usd
FROM silver_transactions t
JOIN silver_orders o ON o.order_id = t.order_id          -- many payments to ONE order: no multiplication
WHERE t.txn_status = 'SUCCESS'
  AND o.order_status NOT IN ('CANCELLED', 'RETURNED');

In [0]:
-- Order counts and revenue are computed separately, then joined by date.
-- This avoids multiplying order counts by the number of payments.
CREATE OR REPLACE TABLE gold_daily_sales
COMMENT 'Daily sales. Grain: one row per order_date. Key: order_date. Revenue per the shared revenue definition (USD).'
AS
WITH orders_day AS (
  SELECT order_date,
         COUNT(*) AS orders_count,
         SUM(CASE WHEN order_status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_orders
  FROM silver_orders GROUP BY order_date
),
rev_day AS (
  SELECT order_date, COUNT(*) AS successful_payments, SUM(revenue_usd) AS revenue_usd
  FROM gold_revenue_base GROUP BY order_date
)
SELECT o.order_date, o.orders_count, o.cancelled_orders,
       ROUND(o.cancelled_orders / o.orders_count, 4)       AS cancellation_rate,
       COALESCE(r.successful_payments, 0)                  AS successful_payments,
       CAST(COALESCE(r.revenue_usd, 0) AS DECIMAL(18,2))   AS revenue_usd
FROM orders_day o
LEFT JOIN rev_day r ON o.order_date = r.order_date;

In [0]:
CREATE OR REPLACE TABLE gold_revenue_breakdown
COMMENT 'Revenue by category and country. Grain: order_date x category x country. Key: those 3 columns. Revenue in USD.'
AS
SELECT order_date, category, country,
       COUNT(*)                          AS successful_payments,
       COUNT(DISTINCT order_id)          AS paid_orders,
       CAST(SUM(revenue_usd) AS DECIMAL(18,2)) AS revenue_usd
FROM gold_revenue_base
GROUP BY order_date, category, country;

In [0]:
-- Reads silver_transactions directly (all statuses), because success rate needs the failures too.
-- success_rate = SUCCESS transactions / all transactions in that group.
CREATE OR REPLACE TABLE gold_payment_success
COMMENT 'Payment performance. Grain: txn_date x payment_method. Key: those 2 columns. success_rate = success_txns / total_txns.'
AS
SELECT CAST(txn_ts AS DATE) AS txn_date, payment_method,
       COUNT(*)                                                  AS total_txns,
       SUM(CASE WHEN txn_status = 'SUCCESS'  THEN 1 ELSE 0 END)  AS success_txns,
       SUM(CASE WHEN txn_status = 'FAILED'   THEN 1 ELSE 0 END)  AS failed_txns,
       SUM(CASE WHEN txn_status = 'PENDING'  THEN 1 ELSE 0 END)  AS pending_txns,
       SUM(CASE WHEN txn_status = 'REFUNDED' THEN 1 ELSE 0 END)  AS refunded_txns,
       ROUND(SUM(CASE WHEN txn_status = 'SUCCESS' THEN 1 ELSE 0 END) / COUNT(*), 4) AS success_rate
FROM silver_transactions
GROUP BY CAST(txn_ts AS DATE), payment_method;

In [0]:
-- Rank every customer by total USD revenue, then keep the top 100.
-- Ties are broken by customer_id so the ranking is deterministic.
CREATE OR REPLACE TABLE gold_top_customers
COMMENT 'Top 100 customers by revenue. Grain: one row per customer_id. Key: customer_id. Revenue in USD.'
AS
SELECT * FROM (
  SELECT customer_id,
         CAST(SUM(revenue_usd) AS DECIMAL(18,2)) AS revenue_usd,
         COUNT(DISTINCT order_id)                AS paid_orders,
         COUNT(*)                                AS successful_payments,
         ROW_NUMBER() OVER (ORDER BY SUM(revenue_usd) DESC, customer_id) AS spend_rank
  FROM gold_revenue_base
  GROUP BY customer_id
)
WHERE spend_rank <= 100;

In [0]:
ALTER TABLE gold_daily_sales ALTER COLUMN order_date          COMMENT 'Date the order was placed';
ALTER TABLE gold_daily_sales ALTER COLUMN orders_count        COMMENT 'Accepted orders placed that day (all statuses)';
ALTER TABLE gold_daily_sales ALTER COLUMN cancelled_orders    COMMENT 'Orders with status CANCELLED that day';
ALTER TABLE gold_daily_sales ALTER COLUMN cancellation_rate   COMMENT 'cancelled_orders / orders_count';
ALTER TABLE gold_daily_sales ALTER COLUMN successful_payments COMMENT 'Eligible SUCCESS payments on those orders';
ALTER TABLE gold_daily_sales ALTER COLUMN revenue_usd         COMMENT 'Revenue in USD per the shared revenue definition';

ALTER TABLE gold_revenue_breakdown ALTER COLUMN order_date          COMMENT 'Date the order was placed';
ALTER TABLE gold_revenue_breakdown ALTER COLUMN category            COMMENT 'Product category (standardised)';
ALTER TABLE gold_revenue_breakdown ALTER COLUMN country             COMMENT 'Order country code (standardised)';
ALTER TABLE gold_revenue_breakdown ALTER COLUMN successful_payments COMMENT 'Eligible SUCCESS payments in the group';
ALTER TABLE gold_revenue_breakdown ALTER COLUMN paid_orders         COMMENT 'Distinct orders with at least one eligible payment';
ALTER TABLE gold_revenue_breakdown ALTER COLUMN revenue_usd         COMMENT 'Revenue in USD per the shared revenue definition';

ALTER TABLE gold_payment_success ALTER COLUMN txn_date       COMMENT 'Date of the transaction (from txn_ts)';
ALTER TABLE gold_payment_success ALTER COLUMN payment_method COMMENT 'Payment method (standardised)';
ALTER TABLE gold_payment_success ALTER COLUMN total_txns     COMMENT 'All accepted transactions in the group';
ALTER TABLE gold_payment_success ALTER COLUMN success_txns   COMMENT 'Transactions with status SUCCESS';
ALTER TABLE gold_payment_success ALTER COLUMN failed_txns    COMMENT 'Transactions with status FAILED';
ALTER TABLE gold_payment_success ALTER COLUMN pending_txns   COMMENT 'Transactions with status PENDING';
ALTER TABLE gold_payment_success ALTER COLUMN refunded_txns  COMMENT 'Transactions with status REFUNDED';
ALTER TABLE gold_payment_success ALTER COLUMN success_rate   COMMENT 'success_txns / total_txns';

ALTER TABLE gold_top_customers ALTER COLUMN customer_id         COMMENT 'Customer identifier (synthetic)';
ALTER TABLE gold_top_customers ALTER COLUMN revenue_usd         COMMENT 'Total revenue in USD per the shared revenue definition';
ALTER TABLE gold_top_customers ALTER COLUMN paid_orders         COMMENT 'Distinct orders with an eligible payment';
ALTER TABLE gold_top_customers ALTER COLUMN successful_payments COMMENT 'Eligible SUCCESS payments';
ALTER TABLE gold_top_customers ALTER COLUMN spend_rank          COMMENT '1 = highest spender; ties broken by customer_id';

In [0]:
-- Fan-out check: rows in the base view must equal distinct txn_ids (each payment counted once).
SELECT COUNT(*) AS base_rows, COUNT(DISTINCT txn_id) AS distinct_txns FROM gold_revenue_base;

-- Total revenue must be identical in the base view, daily sales and the breakdown.
SELECT (SELECT SUM(revenue_usd) FROM gold_revenue_base)      AS base_total,
       (SELECT SUM(revenue_usd) FROM gold_daily_sales)       AS daily_total,
       (SELECT SUM(revenue_usd) FROM gold_revenue_breakdown) AS breakdown_total;

-- Row counts of every Gold output.
SELECT 'gold_daily_sales' AS table_name, COUNT(*) AS row_count FROM gold_daily_sales
UNION ALL SELECT 'gold_revenue_breakdown', COUNT(*) FROM gold_revenue_breakdown
UNION ALL SELECT 'gold_payment_success',   COUNT(*) FROM gold_payment_success
UNION ALL SELECT 'gold_top_customers',     COUNT(*) FROM gold_top_customers;

In [0]:
SELECT * FROM gold_daily_sales ORDER BY order_date LIMIT 10;
SELECT * FROM gold_revenue_breakdown LIMIT 10;
SELECT * FROM gold_payment_success LIMIT 10;
SELECT * FROM gold_top_customers ORDER BY spend_rank LIMIT 10;

In [0]:
SELECT COUNT(*) AS base_rows, COUNT(DISTINCT txn_id) AS distinct_txns FROM gold_revenue_base;

SELECT (SELECT SUM(revenue_usd) FROM gold_revenue_base)      AS base_total,       
       (SELECT SUM(revenue_usd) FROM gold_daily_sales)       AS daily_total,       
       (SELECT SUM(revenue_usd) FROM gold_revenue_breakdown) AS breakdown_total;